# Portfolio MACD Strategy vs Buy & Hold

Backtests a MACD crossover strategy across a weighted 5-stock portfolio.

In [ ]:
import talib as ta
import yfinance as yf
import pandas as pd
import quantstats as qs

In [ ]:
portfolio_weights = {'AAPL':0.2, "NVDA":0.3, "TSLA":0.15, "MSFT":0.2, "META":0.15}

In [ ]:
# download actual prices (not returns) per ticker - MACD needs price levels to build its EMAs
prices = {}
for ticker in portfolio_weights:
    px = yf.Ticker(ticker).history(start='2010-01-01')['Close']
    px.index = px.index.tz_localize(None)
    prices[ticker] = px
prices = pd.DataFrame(prices).dropna()

daily = prices.pct_change().fillna(0)

# run MACD per ticker, not just on AAPL
strategy_returns = pd.DataFrame(index=prices.index)

In [ ]:
for ticker in portfolio_weights:
    macd, signal, _ = ta.MACD(prices[ticker])
    positions = (macd > signal).shift(1, fill_value=False)  # shift(1) = trade next day after crossover
    strategy_returns[ticker] = daily[ticker] * positions

weights = pd.Series(portfolio_weights)


In [ ]:
strategy = (strategy_returns * weights).sum(axis=1).rename('MACD')          # weighted MACD portfolio
buy_hold = (daily * weights).sum(axis=1).rename('Buy & Hold')               # weighted buy & hold, same 5 tickers

sp500 = qs.utils.download_returns("SPY").rename("SP500")  # kept for optional benchmark use


In [ ]:
qs.reports.html(strategy, buy_hold, title="MACD Strategy VS Buy & Hold", output="macd_report.html")